# TabPFN zamiast prototypu: model fundamentowy, zero treningu

Pytanie: czy model meta-trenowany na milionach **syntetycznych** zadań tabelarycznych (TabPFN) rozpoznaje
nowe typologie prania z pięciu przykładów lepiej niż prototyp na surowych cechach? Jeśli tak — to jest
„więcej klas treningowych” bez budowania własnego generatora; jeśli nie — wiemy, że potrzebny jest prior
domenowy (własny generator schematów AML).

Protokół: te same epizody testowe co w bramce (ten sam sampler, ten sam seed, support i query rozłączne po
grupach kont). W każdym epizodzie trzy klasyfikatory dostają **identyczny** support (4 typologie × 5
przypadków = 20 wierszy, 31 cech) i są oceniane na identycznym query:

1. **prototyp** — najbliższa średnia support w standaryzowanych cechach (to, co pracuje w pętli z analitykiem);
2. **TabPFN v2** — `fit` na 20 wierszach, `predict` na query; żadnych wag nie zmieniamy;
3. **regresja logistyczna** na 20 wierszach — klasyczny punkt odniesienia, żeby odróżnić „TabPFN wie coś”
   od „każdy klasyfikator na 20 wierszach radzi sobie lepiej niż średnia”.

Kryterium zapisane przed uruchomieniem: TabPFN wygrywa, jeśli jego trafność minus trafność prototypu
(średnia różnic per epizod) jest ≥ +0.03 na ≥ 4 z 5 foldów. Prognoza p≈0,4. Dodatkowo odnotowujemy,
czy przebija ProtoNet z kampanii (liczby z maila 16.10 wpisane poniżej jako odniesienie).

Co TabPFN **nie** daje: dokładnego rozkładu decyzji na cechy. Prototyp zostaje w pętli z analitykiem;
TabPFN, jeśli wygra, jest „drugą opinią” i osobnym ekranem.

Środowisko: osobny kernel `Python (tabpfn)` (instalacja w README katalogu), żeby nie ruszać środowiska
`fsl`. Model v2 pobiera się przy pierwszym użyciu z publicznego adresu, bez logowania; nowsze wersje
(v2.5+) wymagają akceptacji licencji przez przeglądarkę i tych nie używamy. Licencję wag v2 sprawdź w
repozytorium Prior Labs przed pokazaniem na zewnątrz.


In [ ]:
# --- KONFIGURACJA --------------------------------------------------------------
PROJECT, REGION, BUCKET = "dark-data-discovery", "us-central1", "dark-data-discovery-fsl-data"
DATASET = "samld-case"
SPLIT_URI = ""                 # split.json z kampanii; puste = najnowszy w buckecie; brak = foldy z freeze'u
FOLDY = None                   # None = wszystkie; albo np. [3, 4]
N_EPIZODOW = 300               # 300 -> CI ok. +/-0.012; 1000 = jak bramka, ok. 3x dluzej
N_ESTIMATORS = 4               # ensemble TabPFN (4 = szybki; 8 = domyslny v2)
DEVICE = "auto"                # "cuda" na Workbenchu z GPU; nic nie trenujemy, wiec urzadzenie nie zmienia wyniku
SEED = 0
# ProtoNet z kampanii 16.10 (mail), tylko jako odniesienie w tabeli:
PROTONET_KAMPANIA = {0: 0.8814, 1: 0.8795, 2: 0.8301, 3: 0.9132, 4: 0.6568}


In [ ]:
# --- DANE (nie zmieniaj) ---------------------------------------------------------
import json, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from fsl.config import TrainConfig
from fsl.data.manifest import fetch_manifest, get_geometry
from fsl.data.samld import build_task_samplers, load_frozen_samld

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "configs/pipeline_config_samld.json").exists())
CFG = json.loads((ROOT / "configs/pipeline_config_samld.json").read_text())
BASE = CFG["training"]
WYNIKI_DIR = Path("wyniki")
WYNIKI_DIR.mkdir(exist_ok=True)

ma_split = "split_uri" in TrainConfig.__dataclass_fields__
if ma_split and not SPLIT_URI:
    from google.cloud import storage
    docs = [b for b in storage.Client(project=PROJECT).list_blobs(BUCKET, prefix=f"splits/{DATASET}/")
            if b.name.endswith("/split.json")]
    if docs:
        SPLIT_URI = f"gs://{BUCKET}/{max(docs, key=lambda b: b.updated).name}"
if SPLIT_URI and not ma_split:
    raise SystemExit("ta wersja pakietu fsl nie zna split_uri — zaktualizuj repo (v1.1+) albo ustaw SPLIT_URI = ''")

manifest = fetch_manifest(PROJECT, BUCKET, DATASET)
in_features, _ = get_geometry(manifest, DATASET)


def konfig(fold):
    return TrainConfig(project=PROJECT, region=REGION, bucket=BUCKET, dataset=DATASET, fold=fold,
                       encoder="mlp", in_channels=in_features, image_size=1, seed=SEED,
                       n_way=BASE["n_way"], k_shot=BASE["k_shot"], query=BASE["query"], log_to_vertex=False,
                       **({"split_uri": SPLIT_URI} if ma_split else {}))


bundle, data_sha = load_frozen_samld(konfig(0))
folds = bundle["folds"]
if SPLIT_URI:
    from fsl.data.split import load_split
    split_doc = load_split(SPLIT_URI, PROJECT)
    FLOORS, SPLIT_SHA = split_doc["floors"], split_doc["split_sha"]
else:
    legacy = Path.home() / "data_cache/samld_case_floors.json"
    assert legacy.exists(), "brak split.json i brak ~/data_cache/samld_case_floors.json"
    FLOORS, SPLIT_SHA = json.loads(legacy.read_text())["floors"], ""
FOLDY = list(range(len(folds))) if FOLDY is None else list(FOLDY)
print(f"{DATASET} | split {SPLIT_SHA[:12] or 'freeze'} | dane {data_sha[:12]} | {in_features} cech | foldy {FOLDY}")


In [ ]:
# --- METODY (nie zmieniaj) -------------------------------------------------------
from sklearn.linear_model import LogisticRegression


def podziel_epizod(data, labels, ways, shot, query):
    """Dokladnie jak fast_adapt: sortowanie po etykiecie, pierwsze `shot` wierszy klasy = support, reszta = query."""
    X, y = data.numpy(), labels.numpy()
    order = np.argsort(y, kind="stable")
    X, y = X[order], y[order]
    support = np.zeros(len(y), dtype=bool)
    first = np.arange(ways) * (shot + query)
    for off in range(shot):
        support[first + off] = True
    return X[support], y[support], X[~support], y[~support]


def prototyp_acc(Xs, ys, Xq, yq):
    C = np.stack([Xs[ys == c].mean(0) for c in range(ys.max() + 1)])
    d = ((Xq[:, None, :] - C[None, :, :]) ** 2).sum(-1)
    return float((d.argmin(1) == yq).mean())


def klasyfikator_acc(clf, Xs, ys, Xq, yq):
    clf.fit(Xs, ys)
    return float((np.asarray(clf.predict(Xq)) == yq).mean())


def tabpfn_clf():
    """TabPFN v2: checkpoint bez logowania (publiczny adres). Nowsze wersje sa gated i ich nie uzywamy."""
    from tabpfn import TabPFNClassifier
    from tabpfn.constants import ModelVersion
    return TabPFNClassifier.create_default_for_version(ModelVersion.V2, device=DEVICE, n_estimators=N_ESTIMATORS,
                                                       random_state=SEED)


def lr_clf():
    return LogisticRegression(max_iter=2000, C=1.0)


In [ ]:
# --- POMIAR (nie zmieniaj) -------------------------------------------------------
wiersze, roznice = [], {}
for k in FOLDY:
    t0 = time.time()
    cfg = konfig(k)
    _, _, test_tasks = build_task_samplers(bundle, cfg)     # swiezy sampler = te same epizody co bramka (seed)
    tab = tabpfn_clf()
    acc = {"prototyp": [], "tabpfn": [], "logistic": []}
    for i in range(N_EPIZODOW):
        data, labels = test_tasks.sample()
        Xs, ys, Xq, yq = podziel_epizod(data, labels, cfg.n_way, cfg.k_shot, cfg.query)
        acc["prototyp"].append(prototyp_acc(Xs, ys, Xq, yq))
        acc["tabpfn"].append(klasyfikator_acc(tab, Xs, ys, Xq, yq))
        acc["logistic"].append(klasyfikator_acc(lr_clf(), Xs, ys, Xq, yq))
        if i == 0:
            print(f"fold {k}: pierwszy epizod {time.time() - t0:.1f} s (TabPFN laduje wagi przy pierwszym fit)")
    a = {m: np.array(v) for m, v in acc.items()}
    d = a["tabpfn"] - a["prototyp"]                          # roznica per epizod (ten sam support i query)
    ci = 1.96 * d.std(ddof=1) / np.sqrt(len(d))
    roznice[k] = (float(d.mean()), float(ci))
    wiersze.append({"fold": k, "typologie": ", ".join(folds[k]["test"]), "n_epizodow": N_EPIZODOW,
                    "podloga": FLOORS[str(k)], "protonet_kampania": PROTONET_KAMPANIA.get(k, float("nan")),
                    "prototyp": float(a["prototyp"].mean()), "tabpfn": float(a["tabpfn"].mean()),
                    "logistic": float(a["logistic"].mean()),
                    "tabpfn_minus_prototyp": float(d.mean()), "ci95_roznicy": float(ci)})
    print(f"fold {k} ({time.time() - t0:.0f} s): podloga {FLOORS[str(k)]:.3f} | ProtoNet(kampania) "
          f"{PROTONET_KAMPANIA.get(k, float('nan')):.3f} | prototyp {a['prototyp'].mean():.3f} | "
          f"TabPFN {a['tabpfn'].mean():.3f} | logistic {a['logistic'].mean():.3f} | "
          f"TabPFN-prototyp {d.mean():+.3f} ±{ci:.3f}")

tab = pd.DataFrame(wiersze).set_index("fold")
tab.to_csv(WYNIKI_DIR / "tabpfn.csv")
pd.set_option("display.width", 200)
tab.round(3)


In [ ]:
# --- WERDYKT (nie zmieniaj) ------------------------------------------------------
wygrane = [k for k, (m, ci) in roznice.items() if m >= 0.03]
print("TabPFN - prototyp per fold: " + ", ".join(f"f{k} {m:+.3f}±{ci:.3f}" for k, (m, ci) in roznice.items()))
print(f"foldy z roznica >= +0.03: {len(wygrane)}/{len(roznice)} -> "
      + ("KRYTERIUM SPELNIONE: TabPFN wygrywa z prototypem" if len(wygrane) >= 4 else
         "KRYTERIUM NIESPELNIONE: ogolny prior tabelaryczny nie wystarcza; nastepny krok to prior domenowy"))
lep = [k for k in tab.index if tab.loc[k, "tabpfn"] > tab.loc[k, "protonet_kampania"] + 0.01]
print(f"TabPFN powyzej ProtoNet z kampanii (o > 1 pp): foldy {lep}")
lr_lep = [k for k in tab.index if tab.loc[k, "logistic"] >= tab.loc[k, "tabpfn"] - 0.01]
if lr_lep:
    print(f"UWAGA: regresja logistyczna na 20 wierszach dorownuje TabPFN na foldach {lr_lep} — zysk nie jest "
          "specyficzny dla TabPFN, tylko dla 'klasyfikator zamiast sredniej'")


## Krok 2 (tylko jeśli krok 1 jest blisko progu): dostrojenie na typologiach treningowych

`tabpfn.finetuning.FinetunedTabPFNClassifier(model_version=ModelVersion.V2, device="cuda", epochs=…,
learning_rate=1e-5)` dostraja wagi na jednym zbiorze (X, y): u nas przypadki 9 typologii treningowych foldu
(plus 4 walidacyjne do wczesnego zatrzymania). Potem dostrojony checkpoint ładuje się przez
`TabPFNClassifier(model_path=…)` i ocenia **tym samym** protokołem co wyżej, na typologiach testowych.
Mechanizm ryzyka: dostrajanie na 9 klasach to ten sam wąski prior, który położył ProtoNet; może zepsuć
ogólny prior zamiast go uzupełnić. Prognoza p≈0,3, że dostrojony pobije niedostrojony o ≥ 2 pp. Kryterium
przed uruchomieniem, jak zawsze.
